# 🔍 Finding the suspects

**Time:** ~35 minutes &nbsp;•&nbsp; **Goal:** get from a table of 1,165 cell lines to a single gene worth plotting — using only pandas you already know.

There are no figures in this notebook. That is deliberate.

A plot is an answer to a question, and the quickest way to make a bad figure is to start drawing before you have one. So this notebook does the part that comes first: narrowing a wide table down to a result small enough to be worth a picture. The next notebook draws it.

The question we are chasing:

> **Which gene does skin cancer need that other cancers do not?**

The obvious way to ask it gives the wrong answer. Finding out *why* is the point.

## Our data — 1,165 cancer cell lines

Researchers took every one of these cell lines and used CRISPR to switch off each gene in turn. Each number is a **dependency score**: how badly the cells did without that gene.

- `0` — the cells did not care
- `−1` — losing that gene hurt
- `−2` and below — losing it was close to lethal

`oncotree_lineage` is the tissue the cancer came from. Everything else in `GENES` is a score.

Run this first.

In [ ]:
# Run this first. Every exercise below uses `df`.
import urllib.request
from pathlib import Path

import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/HocheggerLab/y3-bio-python/main/lab07/data/"

DEPMAP = "depmap.csv"
if not Path(DEPMAP).exists():
    urllib.request.urlretrieve(DATA_URL + DEPMAP, DEPMAP)

df = pd.read_csv(DEPMAP)

GENES = ["SOX10", "BRAF", "KRAS", "NRAS", "PAX8", "MYB", "CTNNB1", "EGFR",
         "MDM2", "ESR1", "RAN", "RPL5", "RPS6", "PSMA1", "SNRPD1", "EIF4A3",
         "A1BG", "AADAC"]

print(df.shape)
df[["cell_line_name", "oncotree_lineage", "SOX10", "BRAF", "RAN"]].head()

## Skill 1 — Look before you analyse

Lab 6 again, on a new table. Before asking a dataset anything, find out what one row is, how big it is, and what is in the column you intend to group by.

The four commands, and what each is for:

| | |
|---|---|
| `df.shape` | how many rows and columns |
| `df.head()` | what a row actually looks like |
| `df.info()` | the type of each column, and what is missing |
| `df["col"].value_counts()` | how many rows carry each label |

In [ ]:
# DEMO - what is one row?
print("rows, columns:", df.shape)
print()
print("one row is one cell line:")
print(df.loc[0, ["cell_line_name", "oncotree_primary_disease", "oncotree_lineage"]])

In [ ]:
# DEMO - how many cell lines per cancer type?
counts = df["oncotree_lineage"].value_counts()
print(counts.head(10))
print()
print("lineages in total:", len(counts))
print("skin lines:", counts["Skin"])

Seventy-three skin lines out of 1,165. That is enough to say something about skin, and it is worth knowing *before* you draw a conclusion from it — a lineage with four cell lines in it cannot support one.

Note what `value_counts()` gave you back: a **Series**, indexed by the label. That is why `counts["Skin"]` works.

### 🔍 DIY challenge — know your table

1. How many columns are gene scores, and how many are labels? (`GENES` has the answer, but check it against `df.columns`.)
2. Use `df.info()`. Are there missing values in the gene columns?
3. Which **three** lineages have the most cell lines? Which have fewer than ten?
4. `df["SOX10"].describe()` — what are the minimum, the median and the maximum? Does the median surprise you?

> 💡 A lineage with very few lines is not useless, but every claim you make about it is fragile. Check `counts.tail(10)`.

In [ ]:
# TODO - your turn


## Skill 2 — Keep only the rows you want

A **mask** is a column of `True`/`False`, one per row. Putting it inside `df[...]` keeps the `True` rows.

```python
mask = df["oncotree_lineage"] == "Skin"
skin = df[mask]
```

Read it from the inside out: build the question first, then use it to filter.

In [ ]:
# DEMO - the skin lines, and everything else
skin = df[df["oncotree_lineage"] == "Skin"]
rest = df[df["oncotree_lineage"] != "Skin"]

print("skin:", skin.shape)
print("rest:", rest.shape)
print()
skin[["cell_line_name", "SOX10", "BRAF", "RAN"]].head()

`skin` and `rest` are ordinary DataFrames — everything you know works on them. Together they account for all 1,165 rows, which is worth checking: `len(skin) + len(rest)` should be `len(df)`.

Keeping the complement (`rest`) is the habit to pick up here. Almost every interesting question in this notebook is *skin compared with everything else*, and you cannot ask it with only half the data.

### 🔍 DIY challenge — slice it yourself

1. Make `lung` and `bowel` the same way. How many lines in each?
2. Build a mask for the skin lines that need SOX10 badly (`SOX10 < -1`). How many are there?
3. Two conditions at once: skin lines where `SOX10 < -1` **and** `BRAF < -0.5`. Remember the round brackets around each condition, and `&` rather than `and`.
4. `df["oncotree_lineage"].isin(["Skin", "Lung", "Bowel"])` — what does that give you, and how many rows?

> 💡 Question 3 is the error everyone meets once: `(a) & (b)`, not `a and b`. Try it the wrong way and read the message, so you recognise it later.

In [ ]:
# TODO - your turn


## Skill 3 — Rank the genes

You have 73 skin lines and 18 gene columns. The obvious question: **which gene do these cells need most?**

Take the mean of each gene column, then ask for the most negative values — most negative means most needed.

```python
means = skin.mean(numeric_only=True)
means.nsmallest(5)
```

`numeric_only=True` tells pandas to skip the text columns rather than complain about them. `nsmallest` is `sort_values().head()` in one step.

In [ ]:
# DEMO - which genes do skin cancer lines need most?
skin_mean = skin.mean(numeric_only=True)
skin_mean.nsmallest(5).round(2)

RAN, at about −3.9, well clear of everything else. A gene skin cancer cannot live without.

That looks like a result. Before believing it, do the obvious control: **ask the same question of a cancer that has nothing to do with skin.**

### 🔍 DIY challenge — the control

1. Compute `lung.mean(numeric_only=True).nsmallest(5)`.
2. Do the same for `bowel`, and for `breast`.
3. Compare the four lists. How much do they have in common?
4. Before reading on: what does that tell you about RAN as a **drug target** for melanoma?

In [ ]:
# TODO - your turn


## Skill 4 — 🏆 The trap, and the way out

Here are all four lists side by side.

In [ ]:
# DEMO - the same question, four lineages
for name in ["Skin", "Lung", "Bowel", "Breast"]:
    sub = df[df["oncotree_lineage"] == name]
    top = sub.mean(numeric_only=True).nsmallest(4)
    print(f"{name:8s} n={len(sub):4d}  " + "  ".join(f"{g} {v:.2f}" for g, v in top.items()))

Identical. RAN first every time, then SNRPD1, RPL5, PSMA1.

These are **housekeeping genes** — ribosome, spliceosome, proteasome, nuclear transport. Every cell alive needs them, cancerous or not, skin or lung or bowel. A drug against RAN would work beautifully on melanoma and kill the patient along with it.

So "which gene does skin need most?" was the wrong question. The right one is comparative:

> **Which gene does skin need *more than everything else does*?**

Which is a subtraction. Take the mean in skin, take the mean in the rest, and look at the difference. A large negative difference means: needed in skin, not needed elsewhere.

```python
difference = skin.mean(numeric_only=True) - rest.mean(numeric_only=True)
difference.nsmallest(5)
```

Housekeeping genes cancel out — they are equally essential on both sides, so their difference is near zero. Only the lineage-specific genes survive.

In [ ]:
# DEMO - needed in skin, compared with everywhere else
difference = skin.mean(numeric_only=True) - rest.mean(numeric_only=True)

print("most skin-specific:")
print(difference.nsmallest(5).round(2))
print()
print("RAN, for comparison:", round(difference["RAN"], 2))

**SOX10**, at about −1.06, then BRAF at −0.77 and MDM2 at −0.49.

And RAN — which topped every raw list — comes out near zero. Exactly as it should: equally essential everywhere, so it tells you nothing about skin.

That is the whole move, and it generalises far beyond this dataset. A raw ranking finds what is *strongest*. A difference finds what is *specific*. Only the second one is a drug target.

SOX10 is now a suspect with a number attached. **That** is worth a figure.

### 🏆 DIY finale — your own lineage

1. Pick a lineage with at least 30 cell lines (check `counts`). Build `sub` and its complement `other`.
2. Compute the difference, and take `nsmallest(5)`.
3. Does a plausible lineage gene come out on top? Look it up — `PAX8`, `MYB`, `ESR1`, `CTNNB1` and `EGFR` are all in the table for a reason.
4. Now run the difference the other way round, with `nlargest(5)`. What would a large **positive** difference mean?

> 💡 Question 4 is worth thinking about properly: a positive difference means the gene is needed *less* in your lineage than elsewhere. That is a real and quite different kind of finding.

In [ ]:
# TODO - your turn


## Skill 5 — The loose end

One more thing before you draw anything, because it changes what the figure needs to show.

`skin.mean()` gave you −1.26 for SOX10. A mean is one number standing in for 73, and it can hide a great deal. So ask how many skin lines actually need SOX10.

In [ ]:
# DEMO - does EVERY skin line need SOX10?
sox = skin["SOX10"]

print("skin lines:            ", len(sox))
print("mean SOX10:            ", round(sox.mean(), 2))
print("strongly dependent (< -1):", (sox < -1).sum())
print("barely dependent (> -0.5):", (sox > -0.5).sum())

Eighteen of the seventy-three sit above −0.5 — they get along without SOX10 more or less fine.

So "skin cancer needs SOX10" is true on average and false for a quarter of the cell lines. A bar chart of the mean would state the average and hide the split entirely, which is why the figure you build in the next notebook shows **every cell line as a point**.

Those eighteen lines are also a real piece of melanoma biology, not noise. Keep them in mind — notebook `03` comes back to them.

### 🔍 DIY challenge — meet the exceptions

1. Pull out the skin lines with `SOX10 > -0.5` and print their `cell_line_name`.
2. Sort the skin lines by SOX10 and look at both ends with `.head()` and `.tail()`.
3. Do the SOX10-independent lines also not need BRAF? Compute the mean BRAF score for each group and compare.
4. How would you describe the shape of `skin["SOX10"]` in words, without drawing it? (Then check yourself in the next notebook.)

In [ ]:
# TODO - your turn


## 🔭 Explore further

- Explore any of these genes yourself at <https://depmap.org/portal/>
- SOX10 in melanoma: search for **"lineage addiction"** and **"phenotype switching"** — the eighteen exceptions above are a well-studied phenomenon, not an artefact
- The pandas **comparison with SQL** guide, if group-and-compare feels familiar from somewhere else: <https://pandas.pydata.org/docs/getting_started/comparison/comparison_with_sql.html>

## Recap

You can now, without looking anything up:

- check the shape of a table and the size of each group before trusting any result from it
- filter rows with a mask, and keep the complement so you can compare
- rank columns with `mean(numeric_only=True)` and `nsmallest`
- recognise the housekeeping trap: the strongest dependency is usually the least interesting one
- replace a raw ranking with a **difference** to find what is specific rather than what is strong
- check whether a mean is hiding a split, before choosing how to plot it

You have one gene, one number, and one complication. **Next:** `01_every_plot_type` — draw it.